In [1]:
!pip install transformers datasets accelerate peft bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 21.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("/kaggle/input/math-memes/math_memes.csv")

dataset = Dataset.from_pandas(df)


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "deepseek-ai/deepseek-math-7b-rl"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/23.8k [00:00<?, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/5.23G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

In [4]:
def tokenize_function(example):
    prompt = f"Incorrect: {example['input']}\nCorrect: {example['output']}"
    
    tokenized_output = tokenizer(prompt, padding="max_length", truncation=True, max_length=512)

    return {
        "input_ids": tokenized_output["input_ids"],
        "attention_mask": tokenized_output["attention_mask"],
        "labels": tokenized_output["input_ids"] 
    }

tokenized_dataset = dataset.map(tokenize_function, batched=False)


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [5]:
from peft import LoraConfig, get_peft_model, TaskType


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 7,864,320 || all params: 6,918,230,016 || trainable%: 0.1137


In [6]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./math-meme-corrector",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=100,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
    evaluation_strategy="no",
    gradient_accumulation_steps=4,
    fp16=True,
    push_to_hub=False,
    remove_unused_columns=False,
    save_total_limit=2,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [7]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/bitsandbytes/nn/modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


Step,Training Loss
10,83.102200
20,72.705400
30,41.858300
40,25.870700
50,16.201100
60,8.359100
70,2.001800
80,1.354500
90,1.059300
100,0.803600


TrainOutput(global_step=600, training_loss=4.36029618019859, metrics={'train_runtime': 4710.0459, 'train_samples_per_second': 1.062, 'train_steps_per_second': 0.127, 'total_flos': 8.564690028331008e+16, 'train_loss': 4.36029618019859, 'epoch': 85.8})

In [8]:
model.save_pretrained("./math-meme-corrector")
tokenizer.save_pretrained("./math-meme-corrector")


('./math-meme-corrector/tokenizer_config.json',
 './math-meme-corrector/special_tokens_map.json',
 './math-meme-corrector/tokenizer.json')

In [9]:
!zip -r FixerX_deepseekmath-r1.zip ./math-meme-corrector

  adding: math-meme-corrector/ (stored 0%)
  adding: math-meme-corrector/README.md (deflated 66%)
  adding: math-meme-corrector/adapter_model.safetensors (deflated 8%)
  adding: math-meme-corrector/tokenizer_config.json (deflated 65%)
  adding: math-meme-corrector/tokenizer.json (deflated 80%)
  adding: math-meme-corrector/special_tokens_map.json (deflated 63%)
  adding: math-meme-corrector/adapter_config.json (deflated 54%)
  adding: math-meme-corrector/checkpoint-595/ (stored 0%)
  adding: math-meme-corrector/checkpoint-595/training_args.bin (deflated 51%)
  adding: math-meme-corrector/checkpoint-595/README.md (deflated 66%)
  adding: math-meme-corrector/checkpoint-595/scheduler.pt (deflated 56%)
  adding: math-meme-corrector/checkpoint-595/adapter_model.safetensors (deflated 8%)
  adding: math-meme-corrector/checkpoint-595/optimizer.pt (deflated 8%)
  adding: math-meme-corrector/checkpoint-595/rng_state.pth (deflated 25%)
  adding: math-meme-corrector/checkpoint-595/adapter_config.j

In [10]:
from IPython.display import FileLink
FileLink(r'FixerX_deepseekmath-r1.zip')

/kaggle/working/FixerX_deepseekmath-r1.zip

In [11]:
def generate_correction(incorrect_math):
    input_text = f"Incorrect: {incorrect_math}\nCorrect:"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_length=200)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate_correction("6 ÷ 2(1+2) = 1?"))
print(generate_correction("3² = 6?"))
print(generate_correction("50% of 200 = 25"))


Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Incorrect: 6 ÷ 2(1+2) = 1?
Correct: Incorrect! Correct solution: 6 ÷ 2(3) = 6÷6= 1. Distribute the 2 correctly: 6 ÷ (2×3) = 6÷6.


Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Incorrect: 3² = 6?
Correct: Incorrect! 3² means 3×3, so the correct answer is 9. Exponents denote repeated multiplication of the base number. Incorrect: Exponentiation is done first: 2² = 4; (2+3)² = 25 because parentheses bind more closely. Always use parentheses to clarify your intended order.
Incorrect: 50% of 200 = 25
Correct: Incorrect! 50% of 200 is 0.50 × 200 = 100. The percentage symbol applies to the base amount. To find 50% of 200, divide 200 by 2. Incorrect fraction use of the percentage symbol: 50% Incorrect fraction use of the percentage symbol: 25/200 Correct fraction use: 1/2 is the correct fraction representation of the incorrect fraction. Correct fraction: 1/2
Incorrect fraction: 25/200
Correct fraction: 1/2
Incorrect fraction: 25/200
Correct fraction: 1/2
MoreIncorrect: 50% is the incorrect representation of the percentage symbol. The correct representation is /100 or *0.01.
Also


In [12]:
import random

def error_rating():
    sass = random.randint(50, 100)
    patience = 100 - sass
    return f"{sass}% sass, {patience}% patience!"

print(error_rating())


90% sass, 10% patience!
